In [3]:
"""
Pertemuan 8: UTS Mini Project ML (End-to-End)
=============================================
Dataset: Penguins (Klasifikasi Spesies)
"""

import matplotlib
matplotlib.use("Agg") # Gunakan backend non-interaktif untuk menyimpan plot

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ==========================================================
# 1) Problem & Dataset
# ==========================================================
print("=" * 60)
print("1) PROBLEM & DATASET")
print("=" * 60)
print("Masalah : Klasifikasi spesies penguin (Adelie, Chinstrap, Gentoo)")
print("Tujuan  : Memprediksi spesies penguin berdasarkan karakteristik fisiknya")
print("Sumber  : seaborn.load_dataset('penguins')")

# Load data
df = sns.load_dataset("penguins")

print(f"\nUkuran data awal : {df.shape[0]} baris, {df.shape[1]} kolom")
print(f"Fitur numerik    : bill_length_mm, bill_depth_mm, flipper_length_mm, body_mass_g")
print(f"Fitur kategorikal: island, sex")
print(f"Target           : species")
print(f"\nDistribusi target:")
print(df["species"].value_counts())

# ==========================================================
# 2) EDA (Exploratory Data Analysis)
# ==========================================================
print("\n" + "=" * 60)
print("2) EDA")
print("=" * 60)

# Cek missing value
print("\nMissing values per kolom:")
print(df.isnull().sum())

# Statistik deskriptif
print("\nStatistik deskriptif fitur numerik:")
print(df.describe().round(2))

# Membuat 5 Visualisasi
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Viz 1: Distribusi Target (Species)
sns.countplot(data=df, x="species", hue="species", ax=axes[0, 0], palette="viridis", legend=False)
axes[0, 0].set_title("1. Distribusi Spesies (Target)")

# Viz 2: Distribusi Flipper Length
sns.histplot(data=df, x="flipper_length_mm", kde=True, color="skyblue", ax=axes[0, 1])
axes[0, 1].set_title("2. Distribusi Panjang Sirip (Flipper Length)")

# Viz 3: Scatter plot Bill Length vs Bill Depth berdasarkan spesies
sns.scatterplot(data=df, x="bill_length_mm", y="bill_depth_mm", hue="species", palette="viridis", ax=axes[0, 2])
axes[0, 2].set_title("3. Bill Length vs Bill Depth")

# Viz 4: Boxplot Body Mass by Species
sns.boxplot(data=df, x="species", y="body_mass_g", hue="species", palette="viridis", ax=axes[1, 0], legend=False)
axes[1, 0].set_title("4. Berat Badan (Body Mass) per Spesies")

# Viz 5: Heatmap Korelasi
# (Pilih hanya kolom numerik untuk korelasi)
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
sns.heatmap(df[numeric_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", ax=axes[1, 1])
axes[1, 1].set_title("5. Heatmap Korelasi Fitur Numerik")

axes[1, 2].axis("off") # Kosongkan subplot terakhir

plt.tight_layout()
plt.savefig("uts_penguins_eda.png")
plt.close()
print("\nVisualisasi EDA telah disimpan ke 'uts_penguins_eda.png'")

# 5 Insight
print("\n5 Insight dari EDA:")
print("  1. Terdapat *missing values* pada dataset (terutama di fitur numerik dan sex) yang harus ditangani.")
print("  2. Dataset sedikit *imbalanced*, spesies Adelie memiliki jumlah terbanyak dibandingkan Chinstrap dan Gentoo.")
print("  3. Spesies Gentoo memiliki karakteristik fisik yang sangat berbeda (flipper lebih panjang, massa tubuh lebih berat) dibandingkan Adelie dan Chinstrap.")
print("  4. Fitur 'flipper_length_mm' dan 'body_mass_g' memiliki korelasi positif yang sangat tinggi (0.87).")
print("  5. Terdapat pemisahan (*separability*) yang jelas antara spesies jika melihat plot ukuran paruh (bill length vs depth).")

# ==========================================================
# 3) Preprocessing
# ==========================================================
print("\n" + "=" * 60)
print("3) PREPROCESSING")
print("=" * 60)

# A. Handle Missing Values
# Karena data yang hilang hanya sedikit (sekitar 11 baris), kita drop saja agar data tetap natural.
df_clean = df.dropna().copy()
print(f"Baris setelah drop missing values: {df_clean.shape[0]}")
print("Alasan: Jumlah missing value sangat kecil (< 5%), membuangnya tidak akan menghilangkan banyak informasi.")

# B. Pisahkan X dan y
X = df_clean.drop(columns=["species"])
y = df_clean["species"]

# C. Encoding
# One-Hot Encoding untuk fitur kategorikal (island, sex)
X_encoded = pd.get_dummies(X, columns=["island", "sex"], drop_first=True)
print("\nFitur kategorikal (island, sex) di-encode menggunakan One-Hot Encoding (pd.get_dummies).")
print("Alasan: Model ML berbasis matematika (seperti Logistic Regression) tidak bisa memproses teks string.")

# Encode Target (meskipun sklearn bisa menerima string, lebih rapi jika diubah ke angka)
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# D. Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)
print(f"\nTrain: {X_train.shape[0]} sampel | Test: {X_test.shape[0]} sampel")

# E. Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("StandardScaler diterapkan pada fitur numerik.")
print("Alasan: Untuk menyamakan skala fitur (misal: body_mass ribuan, bill_length puluhan) agar model yang sensitif jarak tidak bias.")

# ==========================================================
# 4) Modeling — minimal 3 model
# ==========================================================
print("\n" + "=" * 60)
print("4) MODELING")
print("=" * 60)

models = {
    "Logistic Regression": LogisticRegression(max_iter=500, random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
}

results = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    pred = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, pred)
    results[name] = acc

# Tabel perbandingan
print("\nTabel Perbandingan Metrik (Akurasi):")
print(f"{'Model':25s}  {'Accuracy':>8s}")
print("-" * 36)
for name, acc in results.items():
    print(f"{name:25s}  {acc:>8.3f}")

# Detail model terbaik
best_name = max(results, key=results.get)
best_model = models[best_name]
pred_best = best_model.predict(X_test_scaled)
print(f"\nDetail model terbaik (berdasarkan akurasi data uji): {best_name}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, pred_best))
print("\nClassification Report:")
print(classification_report(y_test, pred_best, target_names=le.classes_))

# ==========================================================
# 5) Tuning (GridSearchCV)
# ==========================================================
print("=" * 60)
print("5) TUNING")
print("=" * 60)

# Tuning model terbaik (Diasumsikan Random Forest atau Logistic Regression)
if best_name == "Random Forest":
    param_grid = {
        "n_estimators": [50, 100, 200],
        "max_depth": [3, 5, 10, None],
    }
    grid = GridSearchCV(
        RandomForestClassifier(random_state=42),
        param_grid, cv=5, scoring="accuracy", n_jobs=-1
    )
elif best_name == "Logistic Regression":
    param_grid = {
        "C": [0.01, 0.1, 1, 10],
        "solver": ["lbfgs", "liblinear"],
    }
    grid = GridSearchCV(
        LogisticRegression(max_iter=500, random_state=42),
        param_grid, cv=5, scoring="accuracy", n_jobs=-1
    )
else: # Decision Tree
    param_grid = {
        "max_depth": [3, 5, 10, None],
        "min_samples_split": [2, 5, 10],
    }
    grid = GridSearchCV(
        DecisionTreeClassifier(random_state=42),
        param_grid, cv=5, scoring="accuracy", n_jobs=-1
    )

grid.fit(X_train_scaled, y_train)
print(f"Model yang di-tune: {best_name}")
print(f"Best params: {grid.best_params_}")
print(f"Best CV accuracy: {grid.best_score_:.3f}")

tuned_pred = grid.predict(X_test_scaled)
tuned_acc = accuracy_score(y_test, tuned_pred)
print(f"Test accuracy setelah tuning: {tuned_acc:.3f}")

# ==========================================================
# 6) Kesimpulan
# ==========================================================
print("\n" + "=" * 60)
print("6) KESIMPULAN")
print("=" * 60)
print(f"• Model terbaik untuk dataset Penguins adalah {best_name}.")
print(f"• Performa model sangat tinggi (Akurasi ~{tuned_acc:.2f}), menunjukkan bahwa fitur fisik penguin")
print("  (terutama panjang sirip dan bentuk paruh) merupakan prediktor yang sangat kuat untuk membedakan spesies.")
print(f"• Parameter optimal setelah tuning: {grid.best_params_}")
print("• Rekomendasi perbaikan selanjutnya: Karena 'flipper_length_mm' dan 'body_mass_g'")
print("  berkorelasi sangat kuat, kita bisa mencoba teknik reduksi dimensi (seperti PCA)")
print("  atau menghapus salah satu fitur untuk menyederhanakan komputasi model tanpa banyak mengorbankan akurasi.")

print("\n✅ Eksekusi Proyek UTS Selesai.")

1) PROBLEM & DATASET
Masalah : Klasifikasi spesies penguin (Adelie, Chinstrap, Gentoo)
Tujuan  : Memprediksi spesies penguin berdasarkan karakteristik fisiknya
Sumber  : seaborn.load_dataset('penguins')

Ukuran data awal : 344 baris, 7 kolom
Fitur numerik    : bill_length_mm, bill_depth_mm, flipper_length_mm, body_mass_g
Fitur kategorikal: island, sex
Target           : species

Distribusi target:
species
Adelie       152
Gentoo       124
Chinstrap     68
Name: count, dtype: int64

2) EDA

Missing values per kolom:
species               0
island                0
bill_length_mm        2
bill_depth_mm         2
flipper_length_mm     2
body_mass_g           2
sex                  11
dtype: int64

Statistik deskriptif fitur numerik:
       bill_length_mm  bill_depth_mm  flipper_length_mm  body_mass_g
count          342.00         342.00             342.00       342.00
mean            43.92          17.15             200.92      4201.75
std              5.46           1.97              14.0

PERTEMUAN 8 : Project UTS Machine Learning: Klasifikasi Spesies Penguins

Tujuan : membuat model machine learning dari nol (end-to-end). intinya membuat AI agar bisa mengenali spesies cuma dari melihat ukuran badan,panjang sirip,bentuk paruh dan tempat asalnya.

Langkah Kerja :
1. Load data : mengambil dataset bawaan
2. EDA (Exploratory Data Analysis) : lihat isi data,cari insight dan melihat ada yang kosong atau tidak.
3. Preprocessing : membersihkan dan merapihkan data agar siap untuk tahap masuk di algortimanya.
4. Modeling : membandingkan beberapa algoritma untuk melihat yang paling bagus.
5. Tuning : melihat settingan (hyperparameter) model pemenang biar performanya kekanan semua.
6. Kesimpulan : rekap hasil akhir.

Penjelasan Kode :
1. Bagian 1: Problem & Dataset
Kita pakai dataset penguin yang sudah disediain library seaborn. tinggal panggil sns.load_dataset("penguin"), datanya langsung masuk ke dataframe Pandas. disini kita cuma ngeprint ukuran data sama daftar kolomnya agar dapat di bayangkan bentuknya seperti apa.
2. Bagian 2: EDA (Exploratory Data Analysis)
Dibagian ini, sata pakai matplotlib dan seaborn untuk bikin 5 grafik agar mudah baca datanya. Kodenya sengaja saya kasih legend=False dan hue="species" biar tidak ada warning merah seaborn versi terbarunya. semua grafik langsung disimpan jadi satu gambar pada uts_penguins_eda.png.
3. Bagian 3: Preprocessing
ini adalah tahap pembersihan dan merapihkan
 -Drop NA: Kolom yang kosong kita buang aja pakai df.dropna() karena cuma belasan baris, nggak bakal ngaruh banyak.
 -Encoding: Mesin ML nggak ngerti teks (string). Jadi, teks nama pulau sama jenis kelamin kita ubah jadi angka biner pakai pd.get_dummies() (One-Hot Encoding). Nama spesies target juga diubah pakai LabelEncoder jadi 0, 1, dan 2.
 -Split Data: Kita potong datanya jadi 80% buat belajar (train) dan 20% buat ujian (test).
 -Scaling: Karena satuan ukurannya beda-beda (ada yang ribuan gram, ada yang belasan mm), kita ratain skalanya pakai StandardScaler biar komputasinya lancar.
4. Bagian 4: Modeling
kita membandingkan model algoritma ML : logistic regression,decision tree, dan random forest. ketiganya kita suruh belajar dari data model.fit(). terus disuruh menebak data test model.predict(). skornya dibandingin pakai accuracy_score.
5. Bagian 5: Tuning
Kita pakai GridSearchCV buat ngetes berbagai macam kombinasi parameter secara otomatis.

Kesimpulan :
1. Model terbaik buat dataset Penguins adalah (Random Forest) dengan performa akurasi sempurna setelah di- tune.
2. Fitur bentuk fisik (terutama berat badan dan panjang sirip) nyatanya sangat valid buat dipakai membedakan spesies penguin.
3. Saran perbaikan: Ke depannya, karena fitur flipper_length dan body_mass punya fungsi yang mirip dan berkorelasi tinggi, kita bisa pakai teknik Feature Selection buat ngebuang salah satunya supaya sistem modelnya lebih enteng pas memproses data tanpa nurunin akurasi.